# Customer Churn Prediction — end-to-end
A realistic, self-contained machine-learning workflow on a synthetic **telecom
customer churn** dataset (~6,000 customers, imbalanced, missing values,
categorical features).

**Plan**
1. Build the dataset
2. Explore it (EDA)
3. Engineer features
4. Preprocess with a `ColumnTransformer`
5. Compare three model families
6. Tune the best one
7. Final evaluation & a ready-to-paste prediction sample

## 1. Build the dataset
Synthetic but realistic: latent factors from `make_classification` are mapped to interpretable telecom features, categorical columns are derived from them, and missing values are injected.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

RNG = np.random.RandomState(42)

X_lat, y = make_classification(
    n_samples=6000, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.73], flip_y=0.03, class_sep=0.9, random_state=42,
)

def rescale(col, lo, hi):
    c = (col - col.min()) / (col.max() - col.min() + 1e-9)
    return lo + c * (hi - lo)

df = pd.DataFrame({
    "tenure_months":     rescale(X_lat[:, 0], 1, 72).round(0),
    "monthly_charges":   rescale(X_lat[:, 1], 20, 120).round(2),
    "total_charges":     rescale(X_lat[:, 2], 20, 8600).round(2),
    "support_calls":     rescale(X_lat[:, 3], 0, 12).round(0),
    "data_usage_gb":     rescale(X_lat[:, 4], 0.5, 480).round(1),
    "late_payments":     rescale(X_lat[:, 5], 0, 9).round(0),
    "streaming_hours":   rescale(X_lat[:, 6], 0, 90).round(1),
    "avg_call_minutes":  rescale(X_lat[:, 7], 1, 45).round(1),
})

In [ ]:
# Categorical features derived from the remaining latent factors, so they
# genuinely carry signal about churn.
q = pd.qcut(X_lat[:, 8], 3, labels=False)
df["contract_type"] = np.select(
    [q == 0, q == 1], ["month-to-month", "one-year"], default="two-year")

q2 = pd.qcut(X_lat[:, 9], 4, labels=False)
df["payment_method"] = np.select(
    [q2 == 0, q2 == 1, q2 == 2],
    ["electronic-check", "mailed-check", "bank-transfer"],
    default="credit-card")

df["internet_service"] = np.select(
    [df["data_usage_gb"] < 40, df["data_usage_gb"] < 200],
    ["dsl", "fiber"], default="fiber-pro")

df["churn"] = y

In [ ]:
# Inject missing values the way real exports have them (~4%).
mask_tc = RNG.rand(len(df)) < 0.04
mask_du = RNG.rand(len(df)) < 0.03
df.loc[mask_tc, "total_charges"] = np.nan
df.loc[mask_du, "data_usage_gb"] = np.nan

print(f"dataset: {df.shape[0]} rows x {df.shape[1]} cols")
df.head()

## 2. Exploratory analysis

In [ ]:
print("Churn rate:", round(df["churn"].mean() * 100, 1), "%")
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
df.describe().T.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby("contract_type")["churn"].mean().sort_values().plot.bar(
    ax=axes[0], color="#4c72b0", title="Churn rate by contract type")
axes[0].set_ylabel("churn rate")
df["tenure_months"].plot.hist(bins=36, ax=axes[1], color="#55a868",
                              title="Tenure distribution (months)")
plt.tight_layout()
plt.show()

## 3. Feature engineering

In [ ]:
df["charges_per_month"] = (df["total_charges"] / (df["tenure_months"] + 1)).round(2)
df["heavy_support_user"] = (df["support_calls"] >= 6).astype(int)

# Fixed column order: 10 numeric first, then 3 categorical. The
# ColumnTransformer below selects columns BY POSITION so the trained
# pipeline also accepts raw arrays (exactly what a serving endpoint sends).
NUM_COLS = ["tenure_months", "monthly_charges", "total_charges",
            "support_calls", "data_usage_gb", "late_payments",
            "streaming_hours", "avg_call_minutes", "charges_per_month",
            "heavy_support_user"]
CAT_COLS = ["contract_type", "payment_method", "internet_service"]

X = df[NUM_COLS + CAT_COLS]
y = df["churn"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train: {X_train.shape}   test: {X_test.shape}")

## 4. Preprocessing pipeline
Median imputation + scaling for numerics; most-frequent imputation + one-hot encoding for categoricals. Positional selectors keep the fitted pipeline usable on raw arrays at serving time.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM_IDX = list(range(len(NUM_COLS)))                        # positions 0..9
CAT_IDX = list(range(len(NUM_COLS), len(NUM_COLS) + len(CAT_COLS)))  # 10..12

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_IDX),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_IDX),
])

## 5. Compare three model families
Each `fit` is tracked automatically by the platform; the test-set metrics computed right after are attached to the same run.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score,
                             roc_auc_score)

candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42),
}

results = {}
for name, clf in candidates.items():
    model = Pipeline([("prep", preprocess), ("clf", clf)])
    model.fit(X_train.values, y_train)
    pred = model.predict(X_test.values)
    proba = model.predict_proba(X_test.values)[:, 1]
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "model": model,
    }
    print(f"=== {name} ===")
    print(f"  accuracy={results[name]['accuracy']:.4f}  "
          f"f1={results[name]['f1']:.4f}  roc_auc={results[name]['roc_auc']:.4f}")

In [ ]:
summary = pd.DataFrame({k: {m: v[m] for m in ('accuracy', 'f1', 'roc_auc')}
                        for k, v in results.items()}).T
summary.round(4).sort_values("f1", ascending=False)

## 6. Tune the strongest family
A compact manual search on the training split (validation carved out of it), then a final refit on the full training set.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

search_space = [
    {"n_estimators": 400, "max_depth": 14, "min_samples_leaf": 2},
    {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 4},
]

best_params, best_f1 = None, -1.0
for params in search_space:
    m = Pipeline([("prep", preprocess),
                  ("clf", RandomForestClassifier(random_state=42, n_jobs=-1, **params))])
    m.fit(X_tr.values, y_tr)
    val_f1 = f1_score(y_val, m.predict(X_val.values))
    print(f"params={params}  ->  val_f1={val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1, best_params = val_f1, params

print("\nbest params:", best_params)

In [ ]:
final_model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)),
])
final_model.fit(X_train.values, y_train)

final_pred = final_model.predict(X_test.values)
final_proba = final_model.predict_proba(X_test.values)[:, 1]
print("FINAL  accuracy:", round(accuracy_score(y_test, final_pred), 4))
print("FINAL  f1:      ", round(f1_score(y_test, final_pred), 4))
print("FINAL  roc_auc: ", round(roc_auc_score(y_test, final_proba), 4))
print()
print(classification_report(y_test, final_pred, target_names=["stays", "churns"]))

## 7. Final evaluation

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, display_labels=["stays", "churns"], ax=axes[0],
    colorbar=False)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, final_proba, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

In [ ]:
# Top-10 most important features (through the one-hot expansion)
feature_names = final_model.named_steps["prep"].get_feature_names_out()
importances = final_model.named_steps["clf"].feature_importances_
top = sorted(zip(feature_names, importances), key=lambda t: t[1], reverse=True)[:10]

plt.figure(figsize=(8, 4.5))
plt.barh([n for n, _ in top][::-1], [v for _, v in top][::-1], color="#4c72b0")
plt.title("Top-10 feature importances")
plt.tight_layout()
plt.show()

for n, v in top:
    print(f"{n:35s} {v:.4f}")

## 8. Try it on one customer
The row below is ready to paste into the platform's prediction tester once the model is registered and deployed (raw feature order: 10 numerics, then the 3 categorical strings).

In [ ]:
import json

sample = X_test.iloc[0]
instance = [float(sample[c]) if c in NUM_COLS else str(sample[c])
            for c in NUM_COLS + CAT_COLS]
print("prediction-tester payload:")
print(json.dumps([instance]))
print()
print("model says:", "CHURNS" if final_model.predict([instance])[0] == 1 else "STAYS",
      "| churn probability:", round(float(final_model.predict_proba([instance])[0, 1]), 3))

---
**Next steps in the platform**: open **Experiments** — this notebook produced runs
for LogisticRegression, RandomForest (several), and GradientBoosting, grouped and
ranked. Register the best run as a model version, promote it, deploy it, and paste
the payload above into the prediction tester.